# Projet TSA

### Imports

In [132]:
import numpy as np
import cv2

from scipy import signal
from scipy.fftpack import fft
from scipy.linalg import toeplitz

import pandas as pd
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import struct

import msicpe
print(msicpe.__version__)
from msicpe import tsa

1.0.21


*On rappelle que la documentation de la bibliothèque msicpe peut être trouvée en ligne --> __[msicpe](https://cpe.pages.in2p3.fr/msi/toolbox/msicpe.tsa.html)__.*

## Séance 1 - MDFB

### Chargement des données


In [133]:
Fs, s = msicpe.tsa.load_signal('audio')

# réduction de la dimension de s
s = s[:1500]

In [134]:
# propriétés du signal
N = 1500            # nombre de points du signal s
D = N/Fs            # durée du signal s

# vecteur temps
ts = np.linspace(0, (N-1)/Fs, N)

In [135]:
# affichage du signal s
Npts_to_plot = N

df_s = pd.DataFrame({'x': ts[:Npts_to_plot], 'y': s[:Npts_to_plot], 'legende':'Signal audio'})
fig = px.line(df_s, x='x',  y='y', color='legende', labels={'x':'Temps (s)', 'y':''}, title='Signal temporel', width=700)
fig.show()

### Binarisation

In [136]:
# signal binarisé
dtype = 'int'
sb = msicpe.tsa.data2bin(s, dtype=dtype)
Nb = N               # nombre de points du signal sb

# vecteur temps correspondant
d_bin = 10      # frequence d'echantillonnage du signal binaire
tb = np.linspace(0, Nb/d_bin, Nb, endpoint=False)

## **ATTENTION**
L’affichage d’un signal possédant un très grand nombre de points peut faire crasher VSCode.
Aussi, pour chaque signal que vous souhaiterez afficher, pensez à **réduire sa dimension** **dans la dataframe créée pour l’affichage**.

In [137]:
# Affichage de s_b 
Nbits_to_plot = 120

df_sb = pd.DataFrame({'x': tb[:Nbits_to_plot], 'y': sb[:Nbits_to_plot], 'legende':'sb'})
fig2 = px.line(df_sb,x='x',  y='y', color='legende', markers="*", labels={'x': 'Temps (s)' , 'y':'Signal Binarise'}, title= 'Binarisation du Signal' , width=700 )
fig2.show()

### Modulation

In [138]:
# paramètres de modulation
A   = 5
nu0 = 20
nu1 = 40

# vecteur des instants d’échantillonnage du signal modulé s_m
F_mod = 1000
T_mod  = 1/F_mod
N_mod  = int(F_mod/d_bin)

tm_bit = np.linspace(0, N_mod / F_mod, N_mod, endpoint=False)


# signal modulé
s0m = A * np.sin(2 * np.pi * nu0 * tm_bit)
s1m = A * np.sin(2 * np.pi * nu1 * tm_bit)

sm = np.concatenate([s0m if bit==0 else s1m for bit in sb])
# vecteur temps du signal modulé
Nm = len(sm)
tm = np.linspace(0, Nm/F_mod, Nm, endpoint=False)

In [139]:
Nbits_to_plot = 16
Npts_to_plot = Nbits_to_plot * N_mod

# Affichage de s_m
df_sm = pd.DataFrame({'x': tm[:Npts_to_plot] , 'y': sm[:Npts_to_plot] , 'legende': 'Signal Module' })
fig3 = px.line(df_sm, x='x',  y='y', color='legende', labels={'x': 'Temps (s)' , 'y':'Signal Module'}, title= 'Modulation du Signal' , width=700 )
fig3.show()

### Affichages

In [140]:
# Affichage de s_b et s_m sur une seule figure
fig4 = make_subplots(specs=[[{"secondary_y": True}]])

fig41 = px.line(df_sm, x='x',  y='y')
    
fig42 = px.scatter(df_sb, x='x', y='y')
fig42.update_traces(yaxis='y2', marker_color='#cc0000', marker_size=8)

fig4.add_traces(fig41.data + fig42.data)
fig4.update_layout(title='Signaux temporels', width=700,
                    xaxis =dict(title=dict(text='Temps (s)')), 
                    yaxis =dict(title=dict(text='signal modulé sm')), 
                    yaxis2=dict(title=dict(text='signal binaire sb',font=dict(color="#cc0000")), color='#cc0000'),
)
fig4.show()

## Séance 2 - Estimation de la Densité De Probabilité (DDP)

### Analyses _in-silico_

In [141]:
## Fonction de calcul d'histogramme
def histo(x, N=None, M=None):
    x = np.asarray(x).ravel()
    
    if N is not None:
        x = x[:N]
    n = len(x)
    
    # Si nombre de classes M non spécifié, utiliser Freedman-Diaconis
    if M is None:
        q75, q25 = np.percentile(x, [75, 25])
        iqr = q75 - q25
        if iqr == 0:
            sigma = np.std(x, ddof=1)
            Delta = 3.5*sigma/n**(1/3)
        else:
            Delta = 2*iqr/n**(1/3)
        M = max(1, int(np.ceil((x.max() - x.min()) / Delta)))
    else:
        Delta = (x.max() - x.min())/M

    # edges et centroïdes
    edges = np.linspace(x.min(), x.max(), M+1)
    c = (edges[:-1] + edges[1:])/2

    # histogramme non normalisé
    counts, _ = np.histogram(x, bins=edges, density=False)
    
    # histogramme normalisé
    DDPest = counts / (n * Delta)
    
    return DDPest, c, Delta

## Foncton d'affichage
def compareDDP(c, DDPest, DDPth, std_estim, title=None):
    # préparer DataFrame pour DDP théorique et bandes ± std
    df_DDP_th                 = pd.DataFrame({'x': c, 'y': DDPth,    'legende':'DDP_th'})
    df_DDP_th_plus_std_estim  = pd.DataFrame({'x': c, 'y': DDPth+std_estim, 'legende':'DDP_th+std'})
    df_DDP_th_minus_std_estim = pd.DataFrame({'x': c, 'y': DDPth-std_estim, 'legende':'DDP_th-std'})
    
    df_DDP_th = pd.concat([df_DDP_th, df_DDP_th_plus_std_estim, df_DDP_th_minus_std_estim])
    
    fig = px.line(df_DDP_th, x='x',  y='y', title=title,
                  color='legende', color_discrete_sequence=['orangered','black','black'], 
                  line_dash='legende', line_dash_sequence=['solid', 'dash', 'dash'])
    fig.add_bar(x=c, y=DDPest, marker_color='gray', name="DDP estimée")
    fig.show()

In [142]:
# analyse du canal test
b = tsa.canal_test()  # récupération du signal du canal test
n_samples = 5000
b = tsa.canal_test(filtered=False, init=0)
DDPth, c_th, Delta_th = histo(b, N=len(b), M=500)

#### Influence de N

In [143]:
M_fixed = 20
N_values = [100, 500, 1000, 5000]
results_N = []

for N in N_values:
    DDPest, c, Delta = histo(b, N=N, M=M_fixed)
    
    # Interpolation de la DDP "théorique" sur les centroïdes actuels
    DDPth_interp = np.interp(c, c_th, DDPth)
    
    biais = np.mean(DDPest - DDPth_interp)
    var = np.var(DDPest)
    
    results_N.append({'N': N, 'biais': biais, 'variance': var})

df_N = pd.DataFrame(results_N)
print("Effet de N sur la qualité de l'estimation")
print(df_N)

Effet de N sur la qualité de l'estimation
      N     biais  variance
0   100  0.000856  0.016787
1   500  0.000206  0.018218
2  1000  0.001235  0.019759
3  5000 -0.002226  0.019967


#### Influence de $\Delta$

In [144]:
N_fixed = 1000
M_values = [5, 20, 50, 100]
results_M = []

for M in M_values:
    DDPest, c, Delta = histo(b, N=N_fixed, M=M)
    DDPth_interp = np.interp(c, c_th, DDPth)
    
    biais = np.mean(DDPest - DDPth_interp)
    var = np.var(DDPest)
    
    results_M.append({'M': M, 'Delta': Delta, 'biais': biais, 'variance': var})

df_M = pd.DataFrame(results_M)
print("Effet de Δ sur la qualité de l'estimation")
print(df_M)

Effet de Δ sur la qualité de l'estimation
     M     Delta     biais  variance
0    5  1.163409 -0.002348  0.017548
1   20  0.290852  0.001235  0.019759
2   50  0.116341  0.001606  0.020589
3  100  0.058170  0.000534  0.021692


### Analyses _in-situ_

In [145]:
# analyse du canal de transmission réel
canal_id = 1        # identifiant du canal à utiliser
powB = 1            # puissance du bruit (paramètre optionnel)
sm_tr = tsa.transmit(sm, canal_id, powB)  # signal modulé transmis

bruit = sm_tr - sm
moyenne = np.mean(bruit)
variance = np.var(bruit)
ecart_type = np.std(bruit)
puissance = np.mean(bruit**2)

print("Caractérisation statistique du canal de transmission")
print(f"Moyenne du bruit: {moyenne:.4f}")
print(f"Variance du bruit: {variance:.4f}")
print(f"Ecart-type du bruit: {ecart_type:.4f}")
print(f"Puissance du bruit: {puissance:.4f}")

DDPbruit, c_bruit, Delta_bruit = histo(bruit, N=len(bruit), M=50)

df_bruit = pd.DataFrame({'x': c_bruit,'y': DDPbruit,'legende': 'DDP bruit'})
fig = px.bar(df_bruit, x='x', y='y', color='legende', labels={'x':'Amplitude du bruit', 'y':'Densité de probabilité'}, title='DDP du bruit ajouté par le canal', width=700)
fig.show()

Caractérisation statistique du canal de transmission
Moyenne du bruit: -0.0010
Variance du bruit: 1.0002
Ecart-type du bruit: 1.0001
Puissance du bruit: 1.0002


## Séance 3 - Estimation de la Densité Spectrale de Puissance Moyenne

### Analyses _in-silico_

In [146]:
def compareDSP(f, DSPth, DSPbiais, DSPest):
    """Fonction d'affichage compareDSP(f, DSPth, DSPbiais, DSPest) commune aux trois estimateurs, qui trace en dB sur un même graphe.
    Args:
        f (_type_): vecteur de fréquences réduites tel que 0 ≤ f < 0.5
        DSPth (_type_): la DSPM théorique vraie du canal test ΓX (f),
        DSPbiais (_type_): la DSPM théorique obtenue en utilisant l'estimateur ??? du canal test ΓX *W_{N,B}(f),
        DSPest (_type_): votre DSPM estimée dΓ_{???}(f),
    """
    df_th = pd.DataFrame({
        'f': f,
        'y': DSPth[:len(f)],
        'legende': 'DSP théorique'
    })

    df_biais = pd.DataFrame({
        'f': f,
        'y': DSPbiais[:len(f)],
        'legende': 'DSP théorique biaisée'
    })

    df_est = pd.DataFrame({
        'f': f,
        'y': DSPest,
        'legende': 'DSP estimée'
    })

    df = pd.concat([df_th, df_biais, df_est])

    fig = px.line(df, x='f', y='y', color='legende',
                  labels={'f':'Fréquence réduite', 'y':'DSP (dB)'},
                  width=900)

    fig.update_yaxes(range=[-50, 10])  # dynamique conseillée
    fig.show()


##### Estimateur spectral simple

In [147]:
def estimateur_simple(x, nd, N, nfft):
    """
    Estimateur spectral simple (periodogramme)
    x     : séquence d'entrée
    nd    : indice de début
    N     : nombre d'échantillons
    nfft  : taille FFT (puissance de 2 recommandée)

    Retourne :
    f      : fréquences réduites (0 ≤ f < 0.5)
    DSPest : estimation de la DSP en dB
    """

    # extraction du segment étudié
    x_seg = x[nd : nd+N]

    # FFT
    Xf = np.fft.fft(x_seg, n=nfft)

    # periodogramme (normalisation 1/N)
    S = (1/N) * np.abs(Xf)**2

    # garder seulement les fréquences 0 → Nyquist
    half = nfft//2
    S_half = S[:half]
    f = np.linspace(0, 0.5, half, endpoint=False)

    # DSP en dB
    DSPest1 = 10*np.log10(S_half + 1e-15)

    return DSPest1, f

##### Estimateur spectral moyenné

In [148]:
def estimateur_moyenne(x, M, nfft):
    """
    Estimateur spectral moyenné (Bartlett = Welch sans recouvrement)
    
    x     : séquence d'entrée
    M     : taille d’un segment (donc L = N / M segments)
    nfft  : taille FFT (puissance de 2 recommandée)

    Retourne :
    DSPest2 : estimation de la DSP en dB
    f       : fréquences réduites (0 ≤ f < 0.5)
    """

    # Welch avec :
    # - fenêtre RECTANGULAIRE (Bartlett pur)
    # - pas de recouvrement
    # - nperseg = M
    # - nfft défini par l'utilisateur
    # - scaling='density' ⇒ DSP classique
    freqs, S = signal.welch(
        x,
        window='boxcar',
        nperseg=M,
        noverlap=0,
        nfft=nfft,
        return_onesided=True,
        scaling='density'
    )

    # Conversion des fréquences en fréquences réduites (0 ≤ f < 0.5)
    f = freqs / (2 * freqs[-1])

    # passage en dB
    DSPest2 = 10 * np.log10(S + 1e-15)

    return DSPest2, f


##### Estimateur spectral de Welch

In [149]:
def estimateur_welch(x, window, M, Noverlap, nfft, fs=1.0):
    """
    Estimateur de Welch.
    Entrées:
        x        : 1D array, signal
        window   : nom de fenêtre ('hann','hamming','boxcar',...) ou tuple pour get_window
        M        : taille d'un segment (nperseg)
        Noverlap : nombre d'échantillons de recouvrement (noverlap)
        nfft     : taille de la FFT (puissance de 2 conseillée)
        fs       : fréquence d'échantillonnage (pour obtenir freqs en Hz). Par défaut fs=1 -> fréquences réduites.
    Retourne:
        DSPest_dB : estimation de la DSP (en dB) sur 0 <= f < 0.5 (fréquence réduite si fs=1)
        f_reduced : vecteur des fréquences réduites (0 <= f < 0.5)
    Remarque:
        On renvoie les valeurs "one-sided" (positives) et la conversion en dB.
    """
    # utiliser scipy.signal.welch
    freqs, Sxx = signal.welch(
        x,
        fs=fs,
        window=window,
        nperseg=M,
        noverlap=Noverlap,
        nfft=nfft,
        return_onesided=True,
        scaling='density'
    )
    # freqs are in Hz if fs specified. For reduced frequency (0..0.5) with fs=1, freqs already reduced.
    # Convert to reduced frequency in [0,0.5):
    if fs == 1.0:
        f_reduced = freqs  # already reduced
    else:
        f_reduced = freqs / fs  # normalized 0..0.5
    # keep only f < 0.5 (should already be)
    half_mask = f_reduced < 0.5
    f_reduced = f_reduced[half_mask]
    S_half = Sxx[half_mask]

    DSPest3 = 10.0 * np.log10(S_half + 1e-15)
    return DSPest3, f_reduced


### Analyses _in-situ_

In [150]:
# ===============================
# PROGRAMME GLOBAL DES TESTS DSP
# ===============================

# Chargement du signal test
b = tsa.canal_test(filtered=True)

# Théorie 
DSPth, DSPbiais_simple, fth = tsa.sptheo(4096, "simple")
_, DSPbiais_moy, _ = tsa.sptheo(2048, "moyenne")
_, DSPbiais_welch, _ = tsa.sptheo(2048, "welch", "hamming")

# =========================================
# 1) ESTIMATEUR SIMPLE — 1 SEUL GRAPHE
# =========================================
N_simple = 4096
nfft_simple = 4096
nd_simple = 0

DSPest_simple, f_simple = estimateur_simple(b, nd_simple, N_simple, nfft_simple)

compareDSP(
    f_simple,
    DSPth,
    DSPbiais_simple,
    DSPest_simple
)

# =========================================
# 2) ESTIMATEUR MOYENNÉ — 1 SEUL GRAPHE
# =========================================
M_moy = 512      # un M choisi 
nfft_moy = 2048

DSPest_moy, f_moy = estimateur_moyenne(b, M_moy, nfft_moy)

compareDSP(
    f_moy,
    DSPth,
    DSPbiais_moy,
    DSPest_moy
)

# =========================================
# 3) ESTIMATEUR DE WELCH — 1 SEUL GRAPHE
# =========================================
M_welch = M_moy         # souvent = M⋆
Noverlap = M_welch//2   # 50% de recouvrement
nfft_welch = 2048
window = "boxcar"

DSPest_welch, f_welch = estimateur_welch(b, window, M_welch, Noverlap, nfft_welch)

compareDSP(
    f_welch,
    DSPth,
    DSPbiais_welch,
    DSPest_welch
)


In [151]:
# ------------------------------
# Analyse spectrale du canal réel
# ------------------------------

# Paramètres recommandés (Q3.15 : Welch choisi)
M_star = 1024
noverlap = M_star // 2    # 50% recouvrement
nfft = 4096
window = 'hann'
fs = 1.0   # on travaille en fréquence réduite (0..0.5)

bruit = sm_tr - sm

# On prend 'bruit' comme signal d'analyse
bruit = b.astype(float)

# 2) Estimer la DSP du bruit avec Welch (fonction estimateur_welch déjà définie)
DSP_dB, f = estimateur_welch(bruit, window=window, M=M_star, Noverlap=noverlap, nfft=nfft, fs=fs)

# Convertir DSP dB -> puissance linéaire par bin (par unité de fréquence réduite)
PSD_lin = 10**(DSP_dB / 10.0)   # densité de puissance linéaire (same units as input^2 per freq unit)

# 3) Tracer la PSD (Plotly)
df_psd = pd.DataFrame({'f': f, 'PSD_dB': DSP_dB, 'PSD_lin': PSD_lin})
fig = px.line(df_psd, x='f', y='PSD_dB', title='PSD du bruit (Welch)', labels={'f':'fréquence réduite', 'PSD_dB':'PSD (dB)'})
fig.update_yaxes(range=[-80, 10])   # adapter si besoin
fig.show()

# 4) Estimer la bande passante du bruit
# Méthode pratique : chercher fréquence où PSD décroche d'un seuil relatif (ex: -3 dB par rapport au plateau)
# On calcule le "plafond" comme la moyenne du PSD dans la zone la plus élevée (p.ex. 5% supérieur)
idx_sorted = np.argsort(PSD_lin)
# estimation du niveau de bruit de fond (median) et du pic max
noise_floor = np.median(PSD_lin)
peak = np.max(PSD_lin)
# Seuil relatif pour bande passante (ex: niveau à -3 dB sous le max)
th_db = 10*np.log10(peak) - 3.0
th_lin = 10**(th_db / 10.0)

# on prend les fréquences où PSD_lin >= th_lin
mask_band = PSD_lin >= th_lin
if np.any(mask_band):
    f_band = f[mask_band]
    f_low = f_band.min()
    f_high = f_band.max()
else:
    # fallback : prendre bande où PSD_lin > noise_floor * 2
    mask_band = PSD_lin > (noise_floor * 2)
    if np.any(mask_band):
        f_low = f[mask_band].min()
        f_high = f[mask_band].max()
    else:
        f_low, f_high = 0.0, 0.5

bandwidth = f_high - f_low

# Annoter le graphe
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=f, y=DSP_dB, mode='lines', name='PSD (dB)'))
fig2.add_vline(x=f_low, line=dict(color='red', dash='dash'), annotation_text=f"f_low={f_low:.3f}")
fig2.add_vline(x=f_high, line=dict(color='red', dash='dash'), annotation_text=f"f_high={f_high:.3f}")
fig2.update_layout(title="PSD du bruit et bande passante estimée", xaxis_title='Fréquence réduite', yaxis_title='PSD (dB)')
fig2.update_yaxes(range=[-80, 10])
fig2.show()

# 5) Nature du bruit : tests statistiques
# Moments temporels
mean_bruit = np.mean(bruit)
var_bruit = np.var(bruit, ddof=0)
std_bruit = np.sqrt(var_bruit)

# Histogramme + fit gaussien
DDP_bruit, c_bruit, delta_bruit = histo(bruit, N=len(bruit), M=100)

# 6) Puissance moyenne du bruit
# a) en domaine temporel : E[b^2]
power_time = np.mean(bruit**2)

# b) en domaine fréquentiel : intégrale de PSD (Parseval)
# PSD_lin is density per unit reduced frequency. To get total power integrate over 0..0.5:
# use trapezoidal integration on PSD_lin over f
power_freq = np.trapezoid(PSD_lin, f)

# Résultats synthèse
summary = {
    'mean_time': mean_bruit,
    'std_time': std_bruit,
    'variance_time': var_bruit,
    'band_f_low': f_low,
    'band_f_high': f_high,
    'bandwidth': bandwidth,
    'power_time': power_time,
    'power_freq': power_freq
}
df_summary = pd.DataFrame([summary])
display(df_summary)


print(f"Puissance (temps) = {power_time:.6e}, Puissance (fréquence) = {power_freq:.6e} (devraient être proches)")



,mean_time,std_time,variance_time,band_f_low,band_f_high,bandwidth,power_time,power_freq
0,-0.001272,0.532916,0.283999,0.000732,0.133301,0.132568,0.284001,0.283169


Puissance (temps) = 2.840006e-01, Puissance (fréquence) = 2.831691e-01 (devraient être proches)


## Séance 4 - Détection d’un signal noyé dans un bruit

In [152]:
# Transmission
sm_tr = tsa.transmit(sm, canal_id)

#### Débruitage par détection

In [153]:
th =               # vecteur temps d'un symbole

# reponses impulsionnelles des filtres de détection
h0 = 
h1 = 

# filtrage en Fourier
Nfft = 

fm =        # vecteur de fréquences réduites
...

# signaux en sortie des 2 filtres
s0 = 
s1 =

# signal détecté
sd = ...
sd = sd.astype(int)


SyntaxError: invalid syntax (2641988041.py, line 1)

#### Étude de l’effet du bruit de transmission sur la précision

In [ ]:
def error(sd,sb):
    pass
    return e



#### Décodage

In [ ]:
s_tr = 

## Séance 5 - Prédiction AR d’ordre $\textit{M}$

#### Estimation de la fonction d’autocorrélation

In [ ]:
K = ...

Gam = signal.correlate(s_tr.astype(np.float32), s_tr.astype(np.float32), mode='full', method='auto')
Gam /= len(s_tr)
Gam = Gam[len(Gam)//2 - K:len(Gam)//2 + K + 1]
lags = signal.correlation_lags(len(s_tr), len(s_tr), mode="full")
lags = lags[len(lags)//2 - K:len(lags)//2 + K + 1]

#### Identification du modèle AR(M)

In [ ]:
M = 

# matrice du systeme lineaire 
G = toeplitz()

# second membre du systeme lineaire
b = 

# solution du systeme G.Phi = b
Phi = np.linalg.pinv(G) @ b

# coefficients du filtre de Wiener
h = 

# puissance de l'erreur
sigma = 

#### Prédiction Linéaire

In [ ]:
s_hat =

#### Restauration par filtrage causal/anti-causal

In [ ]:
...